# Notebook 1 — Data Ingestion & Indexing

**Objective**: walk the data pipeline that produced the corpus the rest of the project queries — scrape, chunk, embed, index — and validate the committed state.

**Mode**: read-only. We inspect the existing committed artefacts (HTML cache, `data/raw/manifest.csv`, `data/chunks/*`, `chroma_db/`) and run one OpenAI embedding sanity query. We do NOT rebuild the index here, because rebuilding would change the embedding-version footprint that the locked RAGAS evaluation in NB3 was computed against. A guarded "rebuild from scratch" code block is included at the end for reference.

## Reproducibility

In [1]:
import subprocess, sys, importlib.metadata
from datetime import datetime, timezone

def _ver(pkg):
    try: return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError: return "n/a"

print(f"Notebook executed:    {datetime.now(timezone.utc).isoformat(timespec='seconds')}")
print(f"Python:               {sys.version.split()[0]}")
print(f"Git SHA:              {subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()}")
print(f"Git branch:           {subprocess.check_output(['git','rev-parse','--abbrev-ref','HEAD']).decode().strip()}")
print()
print("Key package versions:")
for p in ("anthropic", "openai", "cohere", "chromadb", "langchain",
          "langgraph", "ragas", "tiktoken", "pandas", "boe-rag"):
    print(f"  {p:18s} {_ver(p)}")


Notebook executed:    2026-04-15T23:00:45+00:00
Python:               3.12.12
Git SHA:              5b784c8
Git branch:           main

Key package versions:
  anthropic          0.94.1
  openai             2.31.0
  cohere             6.1.0
  chromadb           1.5.7
  langchain          1.2.15
  langgraph          1.1.6
  ragas              0.4.3
  tiktoken           0.12.0
  pandas             3.0.2
  boe-rag            0.1.0


In [2]:
import os
from dotenv import load_dotenv

load_dotenv()
# Sanity-check that keys are present; do NOT echo any values.
for k in ("OPENAI_API_KEY",):
    if not os.getenv(k):
        raise RuntimeError(f"Missing required env var: {k}")
    print(f"  {k}: present ({len(os.getenv(k))} chars)")


  OPENAI_API_KEY: present (164 chars)


## Scraped corpus

The scraper (`src/boe_rag/scraper/`) downloads MPC minutes, MPRs, FSRs, and speeches from `bankofengland.co.uk`, normalises HTML to plain text with structural markers (`##`, `###`, paragraph numbers, `[BOX START/END]`), and writes a `manifest.csv` with one row per document.

In [3]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
manifest = pd.read_csv(ROOT / "data" / "raw" / "manifest.csv")
print(f"Total documents in manifest: {len(manifest)}")
print()
print("By document type:")
print(manifest.groupby("document_type").size().to_string())
print()
print(f"Total words across corpus: {int(manifest['word_count'].sum()):,}")
print()
print("Sample (first 6 rows):")
display(manifest.head(6))


Total documents in manifest: 23

By document type:
document_type
FSR             2
MPC_minutes     7
MPR             4
speech         10

Total words across corpus: 250,102

Sample (first 6 rows):


,filename,document_type,date,title,source_url,word_count,status,speaker
0,mpc_2025_06.txt,MPC_minutes,2025-06,June 2025 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,4685,ok,NaN
1,mpc_2025_08.txt,MPC_minutes,2025-08,August 2025 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,2630,ok,NaN
2,mpc_2025_09.txt,MPC_minutes,2025-09,September 2025 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,4549,ok,NaN
3,mpc_2025_11.txt,MPC_minutes,2025-11,November 2025 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,3391,ok,NaN
4,mpc_2025_12.txt,MPC_minutes,2025-12,December 2025 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,4652,ok,NaN
5,mpc_2026_02.txt,MPC_minutes,2026-02,February 2026 MPC Minutes,https://www.bankofengland.co.uk/monetary-polic...,3602,ok,NaN


## Chunking

Two parallel chunking strategies, one per pipeline:

- **Baseline** (`src/boe_rag/chunking/base_chunker.py`) — fixed ~500-token chunks via `RecursiveCharacterTextSplitter`, no metadata. The naive control.
- **Enhanced** (`src/boe_rag/chunking/section_chunker.py`) — section-aware splitting on the structural markers, with rich metadata (document_type, date, section_category, speaker for speeches, box_id for box analyses, paragraph_number for MPC minutes).

Both write one JSON file per source document into `data/chunks/{baseline,enhanced}/`.

In [4]:
import json

baseline_dir = ROOT / "data" / "chunks" / "baseline"
enhanced_dir = ROOT / "data" / "chunks" / "enhanced"

baseline_files = sorted(baseline_dir.glob("*.json"))
enhanced_files = sorted(enhanced_dir.glob("*.json"))

baseline_total = sum(json.loads(p.read_text())["total_chunks"] for p in baseline_files)
enhanced_total = sum(json.loads(p.read_text())["total_chunks"] for p in enhanced_files)

print(f"Baseline collection: {len(baseline_files):2d} documents, {baseline_total:4d} chunks total")
print(f"Enhanced collection: {len(enhanced_files):2d} documents, {enhanced_total:4d} chunks total")
print()
print("Sample chunk from MPC June 2025 minutes:")
print()

b_sample = json.loads((baseline_dir / "mpc_2025_06.json").read_text())["chunks"][0]
e_sample = json.loads((enhanced_dir / "mpc_2025_06.json").read_text())["chunks"][0]

print("--- baseline (no metadata, fixed-size split) ---")
print(f"chunk_id: {b_sample['chunk_id']}")
print(f"text ({b_sample['token_count']} tokens, first 250 chars):")
print("    " + b_sample["text"][:250].replace(chr(10), " ") + "...")
print()
print("--- enhanced (section-aware, metadata-tagged) ---")
print(f"chunk_id: {e_sample['chunk_id']}")
print(f"metadata: {e_sample['metadata']}")
print(f"text ({e_sample['token_count']} tokens, first 250 chars):")
print("    " + e_sample["text"][:250].replace(chr(10), " ") + "...")


Baseline collection: 23 documents,  746 chunks total
Enhanced collection: 23 documents,  470 chunks total

Sample chunk from MPC June 2025 minutes:

--- baseline (no metadata, fixed-size split) ---
chunk_id: baseline_mpc_2025_06_001
text (467 tokens, first 250 chars):
    ## Monetary Policy Summary, June 2025  The Monetary Policy Committee (MPC) sets monetary policy to meet the 2% inflation target, and in a way that helps to sustain growth and employment. The MPC adopts a medium-term and forward-looking approach to de...

--- enhanced (section-aware, metadata-tagged) ---
chunk_id: mpc_2025_06_policy_discussion_001
metadata: {'document_type': 'MPC_minutes', 'date': '2025-06', 'section_category': 'policy_discussion', 'speaker': None, 'source_url': 'https://www.bankofengland.co.uk/monetary-policy-summary-and-minutes/2025/june-2025', 'paragraph_range': 'Monetary Policy Summary, June 2025', 'title': 'June 2025 MPC Minutes'}
text (548 tokens, first 250 chars):
    ## Monetary Policy Summary,

## Indexing

Both chunk collections are embedded with OpenAI `text-embedding-3-small` and stored in two ChromaDB collections (`boe_baseline`, `boe_enhanced`) backed by the local `chroma_db/` directory. The collections are created with the `OpenAIEmbeddingFunction` so that subsequent `.query()` calls automatically embed the query with the same model.

In [5]:
import chromadb
from chromadb.utils import embedding_functions
from boe_rag.config import Paths, BASELINE_COLLECTION, ENHANCED_COLLECTION, EMBEDDING_MODEL

client = chromadb.PersistentClient(path=str(Paths.CHROMA_DB))
embed_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=os.environ["OPENAI_API_KEY"],
    model_name=EMBEDDING_MODEL,
)

baseline_col = client.get_collection(BASELINE_COLLECTION, embedding_function=embed_fn)
enhanced_col = client.get_collection(ENHANCED_COLLECTION, embedding_function=embed_fn)

print(f"ChromaDB path:        {Paths.CHROMA_DB}")
print(f"Embedding model:      {EMBEDDING_MODEL}")
print(f"  {BASELINE_COLLECTION:15s}  {baseline_col.count():4d} embedded chunks")
print(f"  {ENHANCED_COLLECTION:15s}  {enhanced_col.count():4d} embedded chunks")


ChromaDB path:        /Users/aangelov/Desktop/boe-rag-project/chroma_db
Embedding model:      text-embedding-3-small
  boe_baseline      746 embedded chunks
  boe_enhanced      470 embedded chunks


### Sanity query

One end-to-end retrieval round-trip on the enhanced collection. Validates that the index is queryable, the embedding round-trip works, and that returned chunks carry expected metadata.

In [6]:
query = "What did the MPC decide about Bank Rate in November 2025?"
res = enhanced_col.query(query_texts=[query], n_results=3)

print(f"Query: {query!r}\n")
print(f"{'rank':>4}  {'chunk_id':40s}  {'similarity':>10s}  {'doc_type':>13s}  {'date':>8s}  preview")
print("-" * 130)
for rank, (cid, doc, dist, meta) in enumerate(zip(
    res["ids"][0], res["documents"][0], res["distances"][0], res["metadatas"][0]
), start=1):
    sim = round(1.0 - dist, 3)
    preview = doc.replace("\n", " ")[:55] + "..."
    dt = meta.get("document_type", "")
    dat = meta.get("date", "")
    print(f"{rank:>4}  {cid:40s}  {sim:>10.3f}  {dt:>13s}  {dat:>8s}  {preview}")


Query: 'What did the MPC decide about Bank Rate in November 2025?'

rank  chunk_id                                  similarity       doc_type      date  preview
----------------------------------------------------------------------------------------------------------------------------------
   1  mpc_2025_12_policy_discussion_001              0.691    MPC_minutes   2025-12  ## Monetary Policy Summary, December 2025  At its meeti...
   2  mpc_2026_02_policy_discussion_005              0.689    MPC_minutes   2026-02  ### MPC members' views  20: Members set out the rationa...
   3  mpc_2025_08_policy_discussion_004              0.677    MPC_minutes   2025-08   weak, with subdued consumption and investment. This pi...


## Validation checklist

Confirmed by the cells above:

- [x] Manifest contains 23 documents across MPC_minutes / MPR / FSR / speech
- [x] Chunk counts non-zero for both collections
- [x] Baseline chunks carry minimal metadata (`chunk_id`, text, token count)
- [x] Enhanced chunks carry rich metadata (`document_type`, `date`, `section_category`, etc.)
- [x] Both ChromaDB collections embedded and queryable with `text-embedding-3-small`
- [x] Sanity query returns relevant MPC chunks with sensible cosine similarity scores

The pipeline is ready to serve queries. Continue to NB2 for the side-by-side baseline vs enhanced demonstration.

## Appendix — How to rebuild from scratch

The committed `chroma_db/` directory is the canonical state used by the locked evaluation in NB3. Rebuilding will re-call OpenAI's embedding API and the resulting embeddings may differ subtly from the committed ones (model version drift, tokeniser updates) — which would invalidate the locked RAGAS results.

To intentionally rebuild the index from scratch (use only when you accept that the eval results will need to be regenerated):

```python
# WARNING: destructive. Wipes both collections and re-embeds all chunks.
# Cost: ~$0.05 + ~2 minutes.
if False:  # set to True to actually run
    from boe_rag.indexing.chroma_store import build_collection
    build_collection("baseline", baseline_dir, BASELINE_COLLECTION, force=True)
    build_collection("enhanced", enhanced_dir, ENHANCED_COLLECTION, force=True)
```